### Init

In [2]:
# pip install transformers spacy torch torchvision
# python -m spacy download en_core_web_sm

import pandas as pd
import spacy
from transformers import pipeline
from IPython.display import display

nlp = spacy.load("en_core_web_sm")
def remove_geography(text):
    if not isinstance(text, str):
        return ""
    doc = nlp(text)
    # remove geopolitical entities and locations
    # why? mggg models were overfitting to geography
    clean_text = " ".join([token.text for token in doc if token.ent_type_ not in ['GPE', 'LOC']])
    return clean_text

# load data and remove names
df = pd.read_csv("data/MOCumulativeAug10.csv")
df['clean_text'] = df['text'].apply(remove_geography)


# load classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# agggregating communities paper labels
candidate_labels = [
    "Agriculture", "Cities", "Community engagement", "Cost of living", 
    "Culture", "Diversity", "Economy and Commerce", "Environment", 
    "Ideology", "Infrastructure", "Elderly", "Family and Children", 
    "K-12 Education", "Named neighborhood", "NIMBY", "Policing", 
    "Poverty", "Recreation and Tourism", "Religion", "Suburbs", 
    "Technology", "University", "Violence", "Vulnerable populations"
]


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

### Label Testimonies

#### Debug

In [8]:
output_ex = classifier(df.iloc[0]['clean_text'], candidate_labels,multi_label=False)
print("Example output:", output_ex)

for i in range(len(output_ex['labels'])):
    print(f"Label: {output_ex['labels'][i]}, Score: {output_ex['scores'][i]:.4f}")

Example output: {'sequence': 'Housing Values , bringing resources to traditionally underserved community and reducing segregation', 'labels': ['Vulnerable populations', 'Community engagement', 'Named neighborhood', 'Diversity', 'Ideology', 'Suburbs', 'Poverty', 'Culture', 'Elderly', 'Family and Children', 'Cities', 'Cost of living', 'Environment', 'University', 'Economy and Commerce', 'Infrastructure', 'Policing', 'Technology', 'Agriculture', 'Religion', 'Recreation and Tourism', 'Violence', 'K-12 Education', 'NIMBY'], 'scores': [0.5210916996002197, 0.23289784789085388, 0.06306028366088867, 0.029685422778129578, 0.018925020471215248, 0.01640518195927143, 0.014331504702568054, 0.013381981290876865, 0.01125803031027317, 0.009088125079870224, 0.008242803625762463, 0.008059716783463955, 0.007091987878084183, 0.006876068655401468, 0.005653802305459976, 0.005100803915411234, 0.004137550480663776, 0.003954275976866484, 0.0038009861018508673, 0.0037814935203641653, 0.003676703665405512, 0.0033

###

In [4]:
def categorize_comment(text):
    if len(text.strip()) < 5:
        return "Unknown"
    result = classifier(text, candidate_labels, multi_label=False)
    
    # top probability label
    return result['labels'][0]

In [5]:





#df.to_csv('MOCategorizedComments.csv')
#df['predicted_category'] = df['clean_text'].apply(categorize_comment)
'''
test = df.head(10)
test['predicted_category'] = test['clean_text'].apply(categorize_comment)


for index, row in test.iterrows():
    print(f"\noriginal text: {row['text']}...")
    print(f"predicted category: {row['predicted_category']}")
'''

'\ntest = df.head(10)\ntest[\'predicted_category\'] = test[\'clean_text\'].apply(categorize_comment)\n\n\nfor index, row in test.iterrows():\n    print(f"\noriginal text: {row[\'text\']}...")\n    print(f"predicted category: {row[\'predicted_category\']}")\n'